<a href="https://colab.research.google.com/github/Lakshya11223/NLP/blob/main/Text_preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Text Preprocessing

### *Data creation*

In [ ]:
!pip install requests


In [ ]:
import requests

headers = {
    "Content-Type": "application/json",
}
data = []
for page in range(1,100):
  request = requests.get(f"https://api.themoviedb.org/3/movie/top_rated?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US&page={page}")
  df = request.json()
  data.append(df)


data

# \/ data
# [
#    {'page': 1,
#   'results': [{'adult': False,
#     'backdrop_path': '/zMwhWailP1WY7sb6AoE6b8ugoy.jpg',
#     'genre_ids': [12, 16, 10751, 14],
#     'id': 1007757,
#     'title': 'Swapped',
#     'original_language': 'en',
#     'original_title': 'Swapped',
#     'overview': 'A small woodland creature and a majestic bird, two natural sworn enemies of the Valley, magically trade places and set off on an adventure of a lifetime to switch back. Their journey soon uncovers a greater threat—one that could endanger not only their species, but the entire valley they call home.',
#     'popularity': 216.7283,
#     'poster_path': '/tHhxWxge06goXU6ZQH1hj7vK8Hd.jpg',
#     'release_date': '2026-05-01',
#     'softcore': False,
#     'video': False,
#     'vote_average': 8.97,
#     'vote_count': 1557},
#   ]}
# {
#     'page': 2
#     'results': [{},{},{}]
# }]

In [ ]:
import pandas as pd

movies = []
genres = requests.get("https://api.themoviedb.org/3/genre/movie/list?api_key=8265bd1679663a7ea12ac168da84d2e8&language=en-US")
genres = genres.json()
genres = genres["genres"]


for page_data in range(len(data)):
  arr = data[page_data]["results"] # this thing you get after seeing the data

  for obj in range(len(arr)):

    name = arr[obj]["title"]
    overview = arr[obj]["overview"]
    genre_ids = arr[obj]["genre_ids"]  # output - [12,15]
    final = []

    # extracting id's name through id
    for id in genre_ids:
      for i in range(len(genres)):
        if genres[i]["id"]==id:
          final.append(genres[i]["name"])



    movies.append({
        "name":name,
        "overview":overview,
        "genres":", ".join(final)
    })

df = pd.DataFrame(movies)
df




,name,overview,genres
0,Swapped,"A small woodland creature and a majestic bird,...","Adventure, Animation, Family, Fantasy"
1,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"Drama, Crime"
2,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama, Crime"
3,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"Science Fiction, Adventure"
4,The Godfather Part II,In the continuing saga of the Corleone crime f...,"Drama, Crime"
...,...,...,...
1975,No Time to Die,Bond has left active service and is enjoying a...,"Action, Thriller, Adventure"
1976,All That Heaven Allows,Two different social classes collide when Cary...,"Drama, Romance"
1977,Ghost Dog: The Way of the Samurai,A Black hitman who models after the samurai of...,"Crime, Drama"
1978,Fantasia,Walt Disney's timeless masterpiece is an extra...,"Animation, Family, Fantasy"


## *Preprocessing*

In [ ]:
df.head()

,name,overview,genres
0,Swapped,"A small woodland creature and a majestic bird,...","Adventure, Animation, Family, Fantasy"
1,The Shawshank Redemption,Imprisoned in the 1940s for the double murder ...,"Drama, Crime"
2,The Godfather,"Spanning the years 1945 to 1955, a chronicle o...","Drama, Crime"
3,Project Hail Mary,Science teacher Ryland Grace wakes up on a spa...,"Science Fiction, Adventure"
4,The Godfather Part II,In the continuing saga of the Corleone crime f...,"Drama, Crime"


In [ ]:
df.shape

(1980, 3)

In [ ]:
# 1st. lower-casing

df['name'] = df['name'].str.lower()

In [ ]:
df['overview'] = df['overview'].str.lower()

In [ ]:
df['genres'] = df['genres'].str.lower()

In [ ]:
df.head()

,name,overview,genres
0,swapped,"a small woodland creature and a majestic bird,...","adventure, animation, family, fantasy"
1,the shawshank redemption,imprisoned in the 1940s for the double murder ...,"drama, crime"
2,the godfather,"spanning the years 1945 to 1955, a chronicle o...","drama, crime"
3,project hail mary,science teacher ryland grace wakes up on a spa...,"science fiction, adventure"
4,the godfather part ii,in the continuing saga of the corleone crime f...,"drama, crime"


In [ ]:
# 2nd. Remoing punctuation

import string
exclude = string.punctuation
def remove_punc1(text):
    return text.translate(str.maketrans('', '', exclude))

In [ ]:
df["genres"] = df["genres"].apply(remove_punc1)
df["overview"]= df["overview"].apply(remove_punc1)
df["name"] = df["name"].apply(remove_punc1)
df.sample(5)

,name,overview,genres
1964,blood and bone,in los angeles an excon takes the underground ...,action crime thriller drama
283,ford v ferrari,american car designer carroll shelby and the b...,drama action history
413,regular show the movie,after a high school lab experiment goes horrib...,animation comedy science fiction adventure
155,piper,a mother bird tries to teach her little one ho...,family animation
1795,the last of the mohicans,in wartorn colonial america in the midst of a ...,action history romance war


In [ ]:
# 3rd removing html tags

import re
def remove_html_tags(text):
    pattern = re.compile('<.*?>')
    return pattern.sub(r'', text)

In [ ]:
df["genres"] = df["genres"].apply(remove_html_tags)
df["overview"]= df["overview"].apply(remove_html_tags)
df["name"] = df["name"].apply(remove_html_tags)
df.sample(5)

,name,overview,genres
839,the man from nowhere,a reclusive pawnshop owner goes on a brutal ra...,action thriller crime
686,fiddler on the roof,in a prerevolutionary russia a poor jewish mil...,drama romance
1359,all that jazz,joe gideon is at the top of the heap one of th...,drama
67,we all loved each other so much,three partisans bound by a strong friendship r...,drama comedy
320,the secret in their eyes,hoping to put to rest years of unease concerni...,mystery thriller drama


In [ ]:
# 4th removing urls

def remove_url(text):
    pattern = re.compile(r'https?://\S+|www\.\S+')
    return pattern.sub(r'', text)

In [ ]:
df["overview"]= df["overview"].apply(remove_url)
df.sample(5)

,name,overview,genres
1872,superman ii the richard donner cut,superman agrees to sacrifice his powers to sta...,science fiction action adventure
639,a woman under the influence,mabel longhetti desperate and lonely is marrie...,drama romance
1607,the bourne ultimatum,bourne is brought out of hiding once again by ...,action drama mystery thriller
870,descendants 3,the teenagers of disneys most infamous villain...,family tv movie adventure fantasy music
1428,the innocents,in a mid19th century essex country house a you...,horror mystery drama


In [ ]:
# 5th removing stop words from description -- a , an , and , the .. etc

from nltk.corpus import stopwords
import nltk
nltk.download('stopwords')

def remove_stopwords(text):
  new_text = []
  for word in text.split():
    if word in stopwords.words('english'):
      new_text.append('')
    else :
      new_text.append(word)

  return " ".join(new_text)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [ ]:
text = "this is an big  picture far from the sun"
remove_stopwords(text)

'   big picture far   sun'

In [ ]:
df["overview"]=df["overview"].apply(remove_stopwords)

In [ ]:
df.head(2)

,name,overview,genres
0,swapped,small woodland creature majestic bird two n...,adventure animation family fantasy
1,the shawshank redemption,imprisoned 1940s double murder wife lo...,drama crime


In [ ]:
# 6th remove emoji if present

import re
def remove_emoji(text):
    emoji_pattern = re.compile("["
                           u"\U0001F600-\U0001F64F"  # emoticons
                           u"\U0001F300-\U0001F5FF"  # symbols & pictographs
                           u"\U0001F680-\U0001F6FF"  # transport & map symbols
                           u"\U0001F1E0-\U0001F1FF"  # flags (iOS)
                           u"\U00002702-\U000027B0"
                           u"\U000024C2-\U0001F251"
                           "]+", flags=re.UNICODE)
    return emoji_pattern.sub(r'', text)

In [ ]:
remove_emoji("Loved the movie. It was 😘😘")

'Loved the movie. It was '

In [ ]:
# Or replace it with name
!pip install emoji


In [ ]:
import emoji
print(emoji.demojize('Python is 🔥'))

Python is :fire:


In [ ]:
emoji.demojize('Loved the movie. It was 😘😘')

'Loved the movie. It was :face_blowing_a_kiss::face_blowing_a_kiss:'

### Tokenizaton

In [2]:
#1. using split function

text = "My name is lakshya. I am an AI engineer"
text.split() # on the basis of words

['My', 'name', 'is', 'lakshya.', 'I', 'am', 'an', 'AI', 'engineer']

In [3]:
text = "My name is lakshya. I am an AI engineer"
text.split('.') # on the basis of sentense

['My name is lakshya', ' I am an AI engineer']

In [ ]:
#2. through regular expression

import re
sent3 = 'I am going to delhi!'
tokens = re.findall("[\w']+", sent3)


In [5]:
tokens

['I', 'am', 'going', 'to', 'delhi']

In [10]:
#3. using library nltk
import nltk
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize, sent_tokenize

sent1 = 'I have a Ph.D in A.I'
sent2 = "We're here to help! mail us at nks@gmail.com"
sent3 = 'A 5km ride cost $10.50'

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [11]:
word_tokenize(sent1)

['I', 'have', 'a', 'Ph.D', 'in', 'A.I']

In [12]:
word_tokenize(sent2) # not handled mail properly

['We',
 "'re",
 'here',
 'to',
 'help',
 '!',
 'mail',
 'us',
 'at',
 'nks',
 '@',
 'gmail.com']

In [13]:
word_tokenize(sent3)

['A', '5km', 'ride', 'cost', '$', '10.50']

In [14]:
#4. using spacy library
import spacy
nlp = spacy.load('en_core_web_sm')

In [15]:
doc1 = nlp(sent1)
doc2 = nlp(sent2)
doc3 = nlp(sent3)

In [16]:
for token in doc3:
    print(token)

A
5
km
ride
cost
$
10.50


In [17]:
for token in doc2:
    print(token)  # it handles mail correctly

We
're
here
to
help
!
mail
us
at
nks@gmail.com


### Stemming & lemmatazisation

In [2]:
from nltk.stem.porter import PorterStemmer

In [3]:
ps = PorterStemmer()
def stem_word(text):
  return " ".join([ps.stem(word) for word in text.split()])

In [4]:
sample = "walk walks walking walked"
stem_word(sample)   # Root wrord extraction

'walk walk walk walk'

In [6]:
text = 'probably my alltime favorite movie a story of selflessness sacrifice and dedication to a noble cause but its not preachy or boring it just never gets old despite my having seen it some 15 or more times in the last 25 years paul lukas performance brings tears to my eyes and bette davis in one of her very few truly sympathetic roles is a delight the kids are as grandma says more like dressedup midgets than children but that only makes them more fun to watch and the mothers slow awakening to whats happening in the world and under her own roof is believable and startling if i had a dozen thumbs theyd all be up for this movie'
stem_word(text)

'probabl my alltim favorit movi a stori of selfless sacrific and dedic to a nobl caus but it not preachi or bore it just never get old despit my have seen it some 15 or more time in the last 25 year paul luka perform bring tear to my eye and bett davi in one of her veri few truli sympathet role is a delight the kid are as grandma say more like dressedup midget than children but that onli make them more fun to watch and the mother slow awaken to what happen in the world and under her own roof is believ and startl if i had a dozen thumb theyd all be up for thi movi'

In [10]:
# lemmatizer

import nltk
nltk.download('punkt_tab')
nltk.download('wordnet')
from nltk.stem import WordNetLemmatizer
wordnet_lemmatizer = WordNetLemmatizer()

# nltk word tokenise

sentence = "He was running and eating at same time. He has bad habit of swimming after playing long hours in the Sun."
punctuations="?:!.,;"

sentence_words = nltk.word_tokenize(sentence)

# removing known punctuation

for word in sentence_words:
    if word in punctuations:
        sentence_words.remove(word)
sentence_words
print("{0:20}{1:20}".format("Word","Lemma"))

# lemmatizer code
for word in sentence_words:
    print ("{0:20}{1:20}".format(word,wordnet_lemmatizer.lemmatize(word,pos='v')))

Word                Lemma               
He                  He                  
was                 be                  
running             run                 
and                 and                 
eating              eat                 
at                  at                  
same                same                
time                time                
He                  He                  
has                 have                
bad                 bad                 
habit               habit               
of                  of                  
swimming            swim                
after               after               
playing             play                
long                long                
hours               hours               
in                  in                  
the                 the                 
Sun                 Sun                 


[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [12]:
!git clone https://github.com/Lakshya11223/NLP.git


Cloning into 'NLP'...
remote: Enumerating objects: 6, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (5/5), done.
Receiving objects: 100% (6/6), 12.39 KiB | 12.39 MiB/s, done.
remote: Total 6 (delta 0), reused 0 (delta 0), pack-reused 0 (from 0)


In [13]:
%cd NLP

/content/NLP


In [14]:
!git status

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
